# Görev 7

In [33]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
%matplotlib inline

In [34]:
!wget -O isimler.txt https://raw.githubusercontent.com/ozymaxx/turkce_dil_verisi/master/tum_isimler.txt

--2026-09-12 22:46:52--  https://raw.githubusercontent.com/ozymaxx/turkce_dil_verisi/master/tum_isimler.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100199 (98K) [text/plain]
Saving to: ‘isimler.txt’

isimler.txt         100%[===================>]  97.85K  --.-KB/s    in 0.002s  

2026-09-12 22:46:52 (46.4 MB/s) - ‘isimler.txt’ saved [100199/100199]



In [35]:
rows = open('isimler.txt', 'r', encoding='utf-8').read().splitlines()

In [36]:
#data cleaning
words = []
for r in rows:
  isim = r.replace('İ', 'i').replace('I','ı')
  isim = isim.lower()
  isim = isim.strip()
  isim = isim.replace('˜', '')
  isim = isim.replace('.', '')
  isim = isim.replace('â', 'a').replace('î', 'i').replace('û', 'u')
  isim = isim.replace('ĩ', 'i').replace('š', 'ü').replace('¦', 'ğ')
  if len(isim) >= 2 and 'w' not in isim and 'x' not in isim and 'q' not in isim:
    words.append(isim)

print(len(words))
words[:8]

12913


['aba', 'abaca', 'abacan', 'abaç', 'abay', 'abayhan', 'abaza', 'abbas']

In [37]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'y', 23: 'z', 24: 'ç', 25: 'ö', 26: 'ü', 27: 'ğ', 28: 'ı', 29: 'ş', 0: '.'}
30


In [38]:
def build_dataset(words):
  block_size = 3
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([74207, 3]) torch.Size([74207])
torch.Size([9196, 3]) torch.Size([9196])
torch.Size([9274, 3]) torch.Size([9274])


In [39]:
n_embd = 10
n_hidden = 200
block_size = 3

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / (n_embd * block_size)**0.5
b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.randn(vocab_size,                      generator=g) * 0

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

12530


In [40]:
@torch.no_grad()
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x]
  embcat = emb.view(emb.shape[0], -1)
  h = torch.tanh(embcat @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 3.406097412109375
val 3.404222011566162


In [41]:
max_steps = 30000
batch_size = 32
lossi = []

for i in range(max_steps):

  # minibatch
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]

  # forward pass
  emb = C[Xb]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Yb)

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  lr = 0.1 if i < 20000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

      0/  30000: 3.4059
  10000/  30000: 1.9133
  20000/  30000: 2.1680


In [42]:
split_loss('train')
split_loss('val')

train 2.0172119140625
val 2.1073923110961914


In [43]:
# sample from the MLP model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size
    while True:
      emb = C[torch.tensor([context])]
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))

    # Bigram Modelden Örnekler
# güsran.
# merslbi.
# te.
# öeri.
# yhimur.

cant.
süzelekminet.
tek.
ergöncemşuyar.
nevayasgan.
ışeri.
haverdoyalitan.
ulutherdağ.
birganima.
mevlim.
koç.
nük.
tandses.
lalen.
ali.
engül.
sertşah.
etekin.
tangülyaryön.
gabiye.


In [44]:
@torch.no_grad()
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x]
  embcat = emb.view(emb.shape[0], -1)
  h = torch.tanh(embcat @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0172119140625
val 2.1073923110961914
